In [ ]:
# Standard library imports
import json
from pathlib import Path
from typing import Optional, Tuple
import warnings

# Third-party imports
import geopandas as gpd
import pandas as pd
import requests
from shapely.geometry import box
import folium
from folium import plugins
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, Button, Output, VBox, HBox
from IPython.display import display, HTML

from pyproj import Transformer, CRS

# Ignore warnings for cleaner output
warnings.filterwarnings('ignore')

# Set up matplotlib for inline plotting
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ All libraries imported successfully!")
print(f"\nLibrary Versions:")
print(f"  GeoPandas: {gpd.__version__}")
print(f"  Pandas: {pd.__version__}")
print(f"  Folium: {folium.__version__}")

✓ All libraries imported successfully!

Library Versions:
  GeoPandas: 1.1.1
  Pandas: 2.3.3
  Folium: 0.20.0


In [3]:
# Create data directory if it doesn't exist
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"✓ Data directory ready: {DATA_DIR.absolute()}")

✓ Data directory ready: c:\Users\Steve\Documents\GitHub\PyDataVT2025\data


In [ ]:
# API Endpoints
TOWN_BOUNDARIES_URL = (
    "https://services1.arcgis.com/BkFxaEFNwHqX3tAw/arcgis/rest/services/"
    "FS_VCGI_OPENDATA_Boundary_BNDHASH_poly_towns_SP_v1/FeatureServer/0/query"
    "?outFields=*&where=1%3D1&f=geojson"
)

GEOLOGY_MAPSERVICE_URL = (
    "https://anrmaps.vermont.gov/arcgis/rest/services/Open_Data/"
    "OPENDATA_ANR_GEOLOGIC_SP_NOCACHE_v2/MapServer/165"
)

GEOLOGY_QUERY_ENDPOINT = f"{GEOLOGY_MAPSERVICE_URL}/query"

BASEMAP_URL = (
    "https://basemaps.arcgis.com/arcgis/rest/services/"
    "World_Basemap_v2/VectorTileServer"
)

# Coordinate Reference Systems
VT_STATE_PLANE = "EPSG:32145"  # Vermont State Plane NAD83 (meters)
WEB_MERCATOR = "EPSG:3857"      # Web Mercator (for tile services)
WGS84 = "EPSG:4326"             # WGS84 (latitude/longitude)

# File paths for cached data
TOWNS_CACHE = DATA_DIR / "towns.geojson"

# Create PyProj Transformer for coordinate conversions
# This transformer converts from WGS84 (lat/lon) to Vermont State Plane (meters)
wgs84_crs = CRS.from_epsg(4326)  # WGS84: latitude, longitude
vt_state_plane_crs = CRS.from_epsg(32145)  # Vermont State Plane: easting, northing

# Create transformer object (always_xy=True means input is longitude, latitude order)
transformer_wgs84_to_vtsp = Transformer.from_crs(wgs84_crs, vt_state_plane_crs, always_xy=True)

# Create reverse transformer for Vermont State Plane to WGS84
transformer_vtsp_to_wgs84 = Transformer.from_crs(vt_state_plane_crs, wgs84_crs, always_xy=True)

print("✓ Configuration complete!")
print(f"\nData Sources:")
print(f"  Towns: VCGI OpenData Portal")
print(f"  Geology: VT Agency of Natural Resources")
print(f"\nCoordinate Systems:")
print(f"  Vermont State Plane: {VT_STATE_PLANE}")
print(f"  WGS84 (Web): {WGS84}")
print(f"\nTransformers:")
print(f"  WGS84 → VT State Plane: transformer_wgs84_to_vtsp")
print(f"  VT State Plane → WGS84: transformer_vtsp_to_wgs84")

In [5]:
# define a function to fetch town boundaries
def fetch_town_boundaries(use_cache: bool = True) -> gpd.GeoDataFrame:
    """
    Fetch Vermont town boundaries from VCGI OpenData portal.
    
    Parameters
    ----------
    use_cache : bool, default True
        If True, load from local cache if available. Otherwise, fetch from API.
    
    Returns
    -------
    gpd.GeoDataFrame
        GeoDataFrame containing Vermont town boundaries with attributes
    
    Notes
    -----
    Data is cached to data/towns.geojson after first download.
    """
    # Check if cached file exists and use_cache is True
    if use_cache and TOWNS_CACHE.exists():
        print(f"📁 Loading towns from cache: {TOWNS_CACHE}")
        gdf = gpd.read_file(TOWNS_CACHE)
        print(f"✓ Loaded {len(gdf)} towns from cache")
        return gdf
    
    # Fetch from API
    print(f"🌐 Fetching town boundaries from VCGI...")
    print(f"   URL: {TOWN_BOUNDARIES_URL[:80]}...")
    
    try:
        # Make HTTP GET request
        response = requests.get(TOWN_BOUNDARIES_URL, timeout=30)
        response.raise_for_status()  # Raise exception for bad status codes
        
        # Parse GeoJSON response
        geojson_data = response.json()
        
        # Convert to GeoDataFrame
        gdf = gpd.GeoDataFrame.from_features(geojson_data['features'])
        
        # Set coordinate reference system
        # VCGI data is in Vermont State Plane (EPSG:32145)
        gdf.set_crs(VT_STATE_PLANE, inplace=True)
        
        print(f"✓ Fetched {len(gdf)} town boundaries")
        
        # Save to cache
        print(f"💾 Saving to cache: {TOWNS_CACHE}")
        gdf.to_file(TOWNS_CACHE, driver='GeoJSON')
        print(f"✓ Cache saved successfully")
        
        return gdf
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching data: {e}")
        raise
    except Exception as e:
        print(f"❌ Error processing data: {e}")
        raise



In [6]:
# Fetch the data
towns_gdf = fetch_town_boundaries()

print(f"\n" + "="*50)
print("TOWN BOUNDARIES LOADED SUCCESSFULLY")
print("="*50)

📁 Loading towns from cache: data\towns.geojson
✓ Loaded 256 towns from cache

TOWN BOUNDARIES LOADED SUCCESSFULLY
✓ Loaded 256 towns from cache

TOWN BOUNDARIES LOADED SUCCESSFULLY


In [7]:
def on_town_selected(town_name: str) -> None:
    """
    Callback function when a town is selected from the dropdown.
    
    Parameters
    ----------
    town_name : str
        Name of the selected town
    """
    global selected_town_data, selected_town_name
    
    # Store the selected town name
    selected_town_name = town_name
    
    # Filter the GeoDataFrame to get the selected town
    selected_town_data = towns_gdf[towns_gdf[town_name_field] == town_name].copy()
    
    if len(selected_town_data) == 0:
        print(f"⚠️  No data found for town: {town_name}")
        return
    
    # Get the first (and should be only) row
    town = selected_town_data.iloc[0]
    
    # Display town information
    print("=" * 70)
    print(f"📍 SELECTED TOWN: {town_name}")
    print("=" * 70)
    
    # Display key attributes
    print(f"\nTown Information:")
    for col in selected_town_data.columns:
        if col != 'geometry':
            value = town[col]
            print(f"  • {col}: {value}")
    
    # Calculate and display area
    # Area is calculated in square meters (since CRS is in meters)
    area_sq_m = town.geometry.area
    area_sq_km = area_sq_m / 1_000_000
    area_acres = area_sq_m / 4046.86
    
    print(f"\nGeographic Properties:")
    print(f"  • Area: {area_sq_km:.2f} km² ({area_acres:.2f} acres)")
    
    # Get bounding box
    bounds = town.geometry.bounds
    print(f"  • Bounding Box (Vermont State Plane, meters):")
    print(f"      Min X: {bounds[0]:,.2f}")
    print(f"      Min Y: {bounds[1]:,.2f}")
    print(f"      Max X: {bounds[2]:,.2f}")
    print(f"      Max Y: {bounds[3]:,.2f}")
    
    # Calculate centroid
    centroid = town.geometry.centroid
    print(f"  • Centroid:")
    print(f"      X: {centroid.x:,.2f}")
    print(f"      Y: {centroid.y:,.2f}")
    
    print("\n" + "=" * 70)
    print("✓ Town data loaded and ready for mapping")
    print("=" * 70)

In [8]:
# Identify the town name column
# Common field names: TOWNNAME, TOWN, NAME, etc.
name_candidates = ['TOWNNAME', 'TOWN', 'NAME', 'Town', 'name']
town_name_field = None

town_name_field = "TOWNNAMEMC"

# Get sorted list of town names
town_names = sorted(towns_gdf[town_name_field].unique())

print(f"\n✓ Found {len(town_names)} Vermont towns")
print(f"\nTown name field: '{town_name_field}'")
print(f"\nSample towns (first 10):")
for name in town_names[:10]:
    print(f"  • {name}")
print(f"  ...")
print(f"  • {town_names[-1]}")


✓ Found 256 Vermont towns

Town name field: 'TOWNNAMEMC'

Sample towns (first 10):
  • Addison
  • Albany
  • Alburgh
  • Andover
  • Arlington
  • Athens
  • Averill
  • Avery's Gore
  • Bakersfield
  • Baltimore
  ...
  • Worcester


In [9]:
# Display first few towns
# Drop geometry column for cleaner display (it's very long)
display_cols = [col for col in towns_gdf.columns if col != 'geometry']

print("\nFirst 5 Towns:")
print("=" * 60)
towns_gdf[display_cols].head()


First 5 Towns:


,OBJECTID,FIPS6,TOWNNAME,TOWNNAMEMC,CNTY,TOWNGEOID,Shape__Area,Shape__Length
0,1,9030.0,CANAAN,Canaan,9,5000911800,8.613393e+07,63085.366335
1,2,11040.0,FRANKLIN,Franklin,11,5001127100,1.058529e+08,42591.256754
2,3,11015.0,BERKSHIRE,Berkshire,11,5001105425,1.085458e+08,42302.494352
3,4,11050.0,HIGHGATE,Highgate,11,5001133025,1.555736e+08,57367.840529
4,5,11060.0,RICHFORD,Richford,11,5001159125,1.118973e+08,42398.623987


In [10]:
# Display column names and types
print("Available Columns:")
print("=" * 60)
for col in towns_gdf.columns:
    dtype = towns_gdf[col].dtype
    if col != 'geometry':
        sample = towns_gdf[col].iloc[0] if len(towns_gdf) > 0 else None
        print(f"  {col:20s} ({dtype}) - Example: {sample}")
    else:
        print(f"  {col:20s} ({dtype})")

Available Columns:
  OBJECTID             (int32) - Example: 1
  FIPS6                (float64) - Example: 9030.0
  TOWNNAME             (object) - Example: CANAAN
  TOWNNAMEMC           (object) - Example: Canaan
  CNTY                 (int32) - Example: 9
  TOWNGEOID            (object) - Example: 5000911800
  Shape__Area          (float64) - Example: 86133926.42295837
  Shape__Length        (float64) - Example: 63085.36633522642
  geometry             (geometry)


In [11]:
# Display basic information
print("Dataset Shape:")
print(f"  Rows (towns): {len(towns_gdf)}")
print(f"  Columns (attributes): {len(towns_gdf.columns)}")

print(f"\nCoordinate Reference System:")
print(f"  {towns_gdf.crs}")
print(f"  Name: {towns_gdf.crs.name}")

print(f"\nGeometry Type:")
print(f"  {towns_gdf.geometry.type.unique()}")

print(f"\nBounding Box (in meters, Vermont State Plane):")
bounds = towns_gdf.total_bounds
print(f"  Min X: {bounds[0]:,.2f}")
print(f"  Min Y: {bounds[1]:,.2f}")
print(f"  Max X: {bounds[2]:,.2f}")
print(f"  Max Y: {bounds[3]:,.2f}")

Dataset Shape:
  Rows (towns): 256
  Columns (attributes): 9

Coordinate Reference System:
  EPSG:32145
  Name: NAD83 / Vermont

Geometry Type:
  ['Polygon' 'MultiPolygon']

Bounding Box (in meters, Vermont State Plane):
  Min X: -73.44
  Min Y: 42.73
  Max X: -71.47
  Max Y: 45.02


In [ ]:
# Create output widget to capture display
output = Output()

print("🎛️  Interactive Town Selector")
print("=" * 70)
print("Select a Vermont town from the dropdown below to view its information.")
print("This will load the town's boundary data for mapping in the next step.")
print("=" * 70)

# Create the dropdown widget
town_dropdown = Dropdown(
    options=town_names,
    value=town_names[0],  # Default to first town
    description='Select Town:',
    style={'description_width': '100px'},
    layout={'width': '400px'}
)

C = town_names[0]

# Wrapper function to clear output before displaying
def update_town_display(town_name):
    with output:
        output.clear_output(wait=True)
        on_town_selected(town_name)

# Use Dropdown.observe instead of interact to avoid duplicate callbacks
def _on_dropdown_change(change):
    # Only respond to user-driven value changes
    if change.get('name') == 'value' and change.get('new') is not None:
        update_town_display(change.get('new'))

# Attach the observer
town_dropdown.observe(_on_dropdown_change, names='value')

# Display the dropdown and the output widget
display(town_dropdown)
display(output)

# The observer will now handle the initial population of the output
# when the dropdown is displayed.

🎛️  Interactive Town Selector
Select a Vermont town from the dropdown below to view its information.
This will load the town's boundary data for mapping in the next step.


Dropdown(description='Select Town:', layout=Layout(width='400px'), options=('Addison', 'Albany', 'Alburgh', 'A…

Output()

In [ ]:
def create_town_map(town_name):
    """
    Create a static map image for the selected Vermont town using ArcGIS MapServer export.
    
    Parameters
    ----------
    town_name : str, optional
        Name of town to display. If None, uses globally selected town.
    
    Notes
    -----
    This uses the ArcGIS REST API 'export' endpoint to generate a static map image
    showing the geology layers for the town's bounding box.
    """
    # Use provided town name or fall back to global selection
    if town_name is None:
        if selected_town_name is None:
            print("⚠️  No town selected. Please select a town first.")
            return None
        town_name = selected_town_name
    
    # Get the town data
    town_data = towns_gdf[towns_gdf[town_name_field] == town_name].copy()
    
    if len(town_data) == 0:
        print(f"⚠️  No data found for town: {town_name}")
        return None
    
    # Get bounding box in WGS84
    bounds = town_data.total_bounds  # [minx, miny, maxx, maxy]
    print(f"   Original Bounding box: {bounds}")

    
    
    
    
    # Add some padding (10% on each side)
    width = bounds[2] - bounds[0]
    height = bounds[3] - bounds[1]
    padding_x = width * 0.1
    padding_y = height * 0.1
    
    bbox_padded = [
        bounds[0] - padding_x,
        bounds[1] - padding_y,
        bounds[2] + padding_x,
        bounds[3] + padding_y
    ]
    
    # Format bbox as comma-separated string
    bbox_str = ','.join(map(str, bbox_padded))
    print(f"   Bounding box with padding: {bbox_str}")
    
    # Construct the MapServer export URL
    base_url = "https://anrmaps.vermont.gov/arcgis/rest/services/Open_Data/OPENDATA_ANR_GEOLOGIC_SP_NOCACHE_v2/MapServer/export"
    
    # Build query parameters
    params = {
        'bbox': bbox_str,
        'bboxSR': '32145',  # Vermont State Plane
        'imageSR': '32145',
        'size': '800,600',
        'dpi': '96',
        'format': 'png32',
        'transparent': 'true',
        'layers': 'show:165',  # Bedrock geology layer
        'f': 'image'
    }
    
    # Make request to get the image
    print(f"🗺️  Generating map for {town_name}...")
    print(f"   Bounding box: {bbox_str}")
    
    try:
        response = requests.get(base_url, params=params, timeout=30)
        response.raise_for_status()
        
        # Display the image
        from IPython.display import Image as IPImage
        display(IPImage(response.content))
        
        print(f"✓ Map displayed for {town_name}")
        print(f"   Image size: 800x600 pixels")
        print(f"   Geology layer: Bedrock geology (Layer 165)")
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching map: {e}")
        raise

# Create and display the map for the selected town
if selected_town_name:
    print(f"Creating map for: {selected_town_name}")
    print("=" * 70)
    create_town_map(selected_town_name)
else:
    print("⚠️  Please select a town from the dropdown above first!")

In [ ]:
# Example: Using the PyProj Transformer
# Let's demonstrate coordinate transformation with an example point

# Example coordinates (Burlington, VT - approximately)
example_lon = -73.2121  # WGS84 longitude
example_lat = 44.4759   # WGS84 latitude

print("🌐 Coordinate Transformation Example")
print("=" * 70)
print(f"\nOriginal Coordinates (WGS84):")
print(f"  Latitude:  {example_lat}°")
print(f"  Longitude: {example_lon}°")

# Transform from WGS84 to Vermont State Plane
# Note: always_xy=True means we pass (longitude, latitude) order
easting, northing = transformer_wgs84_to_vtsp.transform(example_lon, example_lat)

print(f"\nTransformed Coordinates (Vermont State Plane EPSG:32145):")
print(f"  Easting:  {easting:,.2f} meters")
print(f"  Northing: {northing:,.2f} meters")

# Transform back to verify
lon_back, lat_back = transformer_vtsp_to_wgs84.transform(easting, northing)

print(f"\nVerification (Transform back to WGS84):")
print(f"  Latitude:  {lat_back:.6f}°")
print(f"  Longitude: {lon_back:.6f}°")
print(f"  Match: {'✓ Yes' if abs(lat_back - example_lat) < 0.0001 else '✗ No'}")

print("\n" + "=" * 70)
print("ℹ️  Note: Use transformer_wgs84_to_vtsp.transform(lon, lat) for conversions")
print("=" * 70)